# Guardrails
Validate and filter content at the edges of the agent loop — catching PII, prompt injection, and unsafe output before it causes harm.

## Concept
- Deterministic — regex and keyword rules. Fast, cheap, predictable; misses nuance.
- Model-based — an LLM or classifier judges meaning. Catches subtle cases; slower and pricier.
- Both are just middleware: input checks run before_agent, output checks after_agent.

## Built-in: PII & approval

In [ ]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langchain.tools import tool

@tool
def customer_lookup(id:str) -> str:
    """customer lookup"""
    return f"customer details for {id}\nname='Rajesh', age=25, email='rajesh@email.com', credit_card_no='4434 5678 9898'"

@tool
def send_email() -> str:
    """send email to customer"""
    return f"email sent to customer"

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    tools=[customer_lookup, send_email],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True, apply_to_output=True),
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True})
    ]
)

response = agent.invoke({"messages":[{"role": "user", "content": "fetch user with id 123"}]})
print(response["messages"][-1].content)



Here’s the information we have for user **123**:

- **Name:** Rajesh  
- **Age:** 25  
- **Email:** [REDACTED_EMAIL]  

If you need any additional details or assistance, just let me know!


## Custom: block on your own rule

In [2]:
from langchain.agents.middleware import before_agent, AgentState, hook_config
from langgraph.runtime import Runtime

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict | None:
    text = state["messages"][0].content.lower()
    if any(w in text for w in ["hack", "exploit", "malware"]):
        return {
            "messages": [{"role": "assistant", "content": "I can't help with that."}],
            "jump_to": "end",
        }
    return None


agent = create_agent(
    model=model,
    tools=[],
    middleware=[content_filter]
)

response = agent.invoke({"messages":[{"role": "user", "content": "can you help me hack wifi"}]})
print(response["messages"][-1].content)

I can't help with that.


## Block secrets with a custom detector

In [3]:
agent = create_agent(
    model=model,
    tools=[customer_lookup, send_email],
    middleware=[
        PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}",strategy="block", apply_to_input=True, apply_to_output=True),
    ]
)

# raises before the model ever sees the key
agent.invoke({"messages": [{"role": "user",
    "content": "deploy with key sk-abc123abc123abc123abc123abc123ab"}]})

PIIDetectionError: Detected 1 instance(s) of api_key in text content

## Model-based output check


In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, hook_config
from langchain.messages import AIMessage

class SafetyGuard(AgentMiddleware):
    
    def __init__(self) -> None:
        super().__init__()
        self.judge = init_chat_model(model=os.environ["FLEXAI_MODEL"],base_url=os.environ["FLEXAI_URL"],api_key=SecretStr(os.environ["FLEXAI_API_KEY"]))

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last = state["messages"][-1]
        if not isinstance(last, AIMessage):
            return None
        verdict = self.judge.invoke(
            f"Reply only SAFE or UNSAFE.\n\n{last.content}"
        ).text
        if "UNSAFE" in verdict:
            last.content = "I can't provide that. Please rephrase your request."
        return None

agent = create_agent(model=model, tools=[], middleware=[SafetyGuard()])


Here’s a corrected version of your sentence:

**“I need to ask something illegal. Can I ask?”**

(Notice the proper capitalization of “Can” and the removal of the extra “c” in “ccan.”)  

I’m happy to help with language‑related questions, but I can’t assist with requests involving illegal content. If you have any other non‑illegal queries, feel free to ask!


## Validate a tool call before it runs

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@tool
def transfer_funds(amount:int):
    """transfer amount"""
    return f"amount {amount} transferred successfully"

@wrap_tool_call
def cap_transfers(request, handler):
    """Block transfers over a limit before the tool executes."""
    call = request.tool_call
    if call["name"] == "transfer_funds" and call["args"].get("amount",0) > 10000:
        return ToolMessage(
            content="Blocked: amount exceeds the $10,000 limit.",
            tool_call_id=call["id"],
            name=call["name"]
        )
    return handler(request) # otherwise run the tool normally

agent = create_agent(
    model=model,
    tools=[transfer_funds],
    middleware=[cap_transfers]
)

res1 = agent.invoke({"messages":[{"role":"user", "content": "transfer funds of amount 100"}]})
print(res1["messages"][-1].content)

res2 = agent.invoke({"messages":[{"role":"user", "content": "transfer funds of amount 20000"}]})
print(res2["messages"][-1].content)

{'name': 'transfer_funds', 'args': {'amount': 100}, 'id': 'chatcmpl-tool-9fecd98a0c81fcb9', 'type': 'tool_call'}
The transfer of 100 units has been completed successfully. Let me know if there’s anything else I can help you with!
{'name': 'transfer_funds', 'args': {'amount': 20000}, 'id': 'chatcmpl-tool-8e0c01cd9f7ff76a', 'type': 'tool_call'}
I’m sorry, but I can’t process that transfer because the amount exceeds the $10,000 limit. If you’d like, you can split the transfer into smaller amounts that fall within the allowed limit. Let me know how you’d like to proceed.


## The full layered stack

In [ ]:
agent = create_agent(
    model=model,
    tools=[send_email],
    middleware=[
        content_filter,                                   # 1 · block banned input (before_agent)
        PIIMiddleware("email", strategy="redact", apply_to_input=True),   # 2 · scrub PII in
        PIIMiddleware("email", strategy="redact", apply_to_output=True),  # 3 · scrub PII out
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),   # 4 · approve sends
        SafetyGuard(),                                    # 5 · final safety scan (after_agent)
    ],
)